# 環境変数

In [ ]:
import os
os.environ["ERG_DATA_DIR"] = "/mnt/j/observation_data/"

# 3dfluxデータを時間軸に焼き直す

In [ ]:
import pyspedas as psp
import pytplot as pt
import numpy as np
import xarray as xr

pt.del_data('*')

time_range_full = ['2017-11-15/16:00:00', '2017-11-15/17:00:00']
psp.erg.lepi(time_range_full, datatype='3dflux', get_support_data=True, no_update=True, version='v03_00')

# 解析対象の狭い時間窓
time_range = ['2017-11-15/16:10:00', '2017-11-15/16:25:00']

# flux3d本体 (dims = time, v1(energy), v2(channel), v3(phase))  # [#/cm2/sr/sec/keV]
flux3d = pt.data_quants['erg_lepi_l2_3dflux_FPDU'].sel(
    time=slice(*time_range)
)

# 各軸の座標を取り出しておく
time_ax     = flux3d.time.values        # shape = (T,)
energy_ax   = flux3d.v1.values          # (E,)
channel_ax  = flux3d.v2.values          # (C,)
spin_ax     = flux3d.v3.values          # (S,)

time_num, energy_num, channel_num, spin_num = len(time_ax), len(energy_ax), len(channel_ax), len(spin_ax)   # T, E, C, S

# 1 spin time = 8 sec
# 1 spin phase time = 0.5 sec
# 1 energy time step = 15625 μsec
spin_offset_ns      = np.arange(spin_num, dtype='timedelta64[ns]') * 500_000_000        # 0.5 sec = 500,000,000 nsec
energy_offset_ns    = (np.arange(energy_num, dtype='int64') * 15_625_000 + 7_812_500).astype('timedelta64[ns]')

offset_ns           = energy_offset_ns[:, None] + spin_offset_ns    # (E, 1) + (S) -> (E, S)

flux_E_TS_C = flux3d.transpose('v1_dim', 'time', 'v3_dim', 'v2_dim').values.reshape(energy_num, time_num*spin_num, channel_num)

# xarray.DataArrayをエネルギーごとに生成
flux_data_arrays = {}
for energy_i in range(energy_num):
    time_flat = (time_ax[:, None] + offset_ns[energy_i][None, :]).reshape(-1) # ((T, 1) + (1, S) -> (T, S)).reshape(-1) -> (TxS,)

    flux_data_arrays[energy_i] = xr.DataArray(
        flux_E_TS_C[energy_i],
        dims=['time', 'channel'],
        coords={
            'time':         time_flat,
            'channel':      channel_ax,
            'energy_keV':   energy_ax[energy_i]
        },
        attrs=flux3d.attrs,
        name=f'flux_energy_{energy_i}'
    )

print(flux_data_arrays)

fidu_angle_dict     = pt.data_quants['erg_lepi_l2_3dflux_FIDU_Angle_sga']
fidu_angle  = fidu_angle_dict['data'].astype(float)     # shape (2, 3, 16)
AZ_deg_mid = fidu_angle[0, 1, :channel_num]      # shape (C,)
# AZ_deg_mid =  [ 78.75  56.25  33.75  11.25 -11.25 -33.75 -56.25 -78.75]

theta_sga = AZ_deg_mid                  # (C,)
varphi_sga = -90. * np.ones(spin_num)   # (S,)

# (TxS, C)の2次元配列を生成
theta_sga_time = np.tile(theta_sga, (time_num*spin_num, 1))   # (TxS, C)
varphi_sga_times = np.tile(varphi_sga, (time_num, 1)).reshape(-1, 1)   # (TxS, 1)
varphi_sga_time = np.tile(varphi_sga_times, (1, channel_num))   # (TxS, C)

angle_sga_time = np.stack((theta_sga_time, varphi_sga_time), axis=2)   # (TxS, C, 2)

# theta_sga, varphi_sga -> Vx_sga, Vy_sga, Vz_sga
vector_sga_time = np.zeros((time_num*spin_num, channel_num, 3))   # (TxS, C, 3)
vector_sga_time[:, :, 0] = np.cos(np.radians(angle_sga_time[:, :, 0])) * np.cos(np.radians(angle_sga_time[:, :, 1]))
vector_sga_time[:, :, 1] = np.cos(np.radians(angle_sga_time[:, :, 0])) * np.sin(np.radians(angle_sga_time[:, :, 1]))
vector_sga_time[:, :, 2] = np.sin(np.radians(angle_sga_time[:, :, 0]))

v_unit_vector_sga_energy_channel_list = {}
for energy_i in range(energy_num):
    time_flat = (time_ax[:, None] + offset_ns[energy_i][None, :]).reshape(-1)

    for channel_i in range(channel_num):
        v_unit_vector_sga_energy_channel_list[energy_i, channel_i] = xr.DataArray(
            vector_sga_time[:, channel_i, :],
            dims=['time', 'xyz'],
            coords={
                'time': time_flat,
                'xyz': ['x', 'y', 'z'],
                'channel': channel_ax[channel_i],
                'energy_keV':   energy_ax[energy_i]
            },
            name=f'v_unit_vector_sga_{energy_i}_{channel_i}'
        )
        dot_ = (v_unit_vector_sga_energy_channel_list[energy_i, channel_i] * v_unit_vector_sga_energy_channel_list[energy_i, channel_i]).sum(dim='xyz')
        print(f'v_unit_vector_sga_energy_channel_list[{energy_i}, {channel_i}] = ', np.nanmin(dot_), np.nanmax(dot_), np.nanmean(dot_))#v_unit_vector_sga_energy_channel_list[energy_i, channel_i])

# SGA座標系→SGI座標系に変換

In [ ]:
import pyspedas as psp
import pytplot as pt
import xarray as xr

v_unit_vector_sgi_energy_channel_list = {}
for energy_i in range(energy_num):
    for channel_i in range(channel_num):
        pt.store_data(f'vector_sga_{energy_i}_{channel_i}', data={'x': v_unit_vector_sga_energy_channel_list[energy_i, channel_i].time, 'y': v_unit_vector_sga_energy_channel_list[energy_i, channel_i].values})
        # SGI座標系に変換
        psp.projects.erg.sga2sgi(name_in=f'vector_sga_{energy_i}_{channel_i}', name_out=f'vector_sgi_{energy_i}_{channel_i}')
        _data   = pt.data_quants[f'vector_sgi_{energy_i}_{channel_i}'].rename({"v_dim": "xyz"})
        #_data_2 = (_data * _data).sum(dim='xyz')
        _data_unit  = _data #/ np.sqrt(_data_2)
        v_unit_vector_sgi_energy_channel_list[energy_i, channel_i] = xr.DataArray(
            _data_unit.data,
            dims=['time', 'xyz'],
            coords={
                'time': _data.time,
                'xyz': ['x', 'y', 'z'],
                'channel': channel_ax[channel_i],
                'energy_keV':   energy_ax[energy_i]
            },
            name=f'v_unit_vector_sgi_{energy_i}_{channel_i}'
        )

for energy_i in range(energy_num):
    for channel_i in range(channel_num):
        _dot    = (v_unit_vector_sgi_energy_channel_list[energy_i, channel_i] * v_unit_vector_sgi_energy_channel_list[energy_i, channel_i]).sum(dim='xyz')
        print(f'v_unit_vector_sgi_energy_channel_list[{energy_i}, {channel_i}] = ', np.nanmin(_dot), np.nanmax(_dot), np.nanmean(_dot))#v_unit_vector_sgi_energy_channel_list[energy_i, channel_i])

In [ ]:
dot_test = (v_unit_vector_sgi_energy_channel_list[25, 3] * v_unit_vector_sgi_energy_channel_list[25, 3]).sum(dim='xyz')

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(10, 4))
dot_test.plot(ax=ax)
ax.set_xlabel("Time")
ax.set_ylabel("Data")
ax.set_title("dot(unit_vector_sgi) vs time")
plt.show()

# SGA座標系→DSI座標系に変換

In [ ]:
import pyspedas as psp
import pytplot as pt
import xarray as xr

v_unit_vector_dsi_energy_channel_list = {}
for energy_i in range(energy_num):
    for channel_i in range(channel_num):
        # dsi座標系に変換
        psp.projects.erg.sgi2dsi(name_in=f'vector_sgi_{energy_i}_{channel_i}', name_out=f'vector_dsi_{energy_i}_{channel_i}')
        _data   = pt.data_quants[f'vector_dsi_{energy_i}_{channel_i}'].rename({"v_dim": "xyz"})
        #_data_2 = (_data * _data).sum(dim='xyz')
        _data_unit  = _data #/ np.sqrt(_data_2)
        v_unit_vector_dsi_energy_channel_list[energy_i, channel_i] = xr.DataArray(
            _data_unit.data,
            dims=['time', 'xyz'],
            coords={
                'time': _data.time,
                'xyz': ['x', 'y', 'z'],
                'channel': channel_ax[channel_i],
                'energy_keV':   energy_ax[energy_i]
            },
            name=f'v_unit_vector_dsi_{energy_i}_{channel_i}'
        )

for energy_i in range(energy_num):
    for channel_i in range(channel_num):
        dot_    = (v_unit_vector_dsi_energy_channel_list[energy_i, channel_i] * v_unit_vector_dsi_energy_channel_list[energy_i, channel_i]).sum(dim='xyz')
        print(f'v_unit_vector_dsi_energy_channel_list[{energy_i}, {channel_i}] = ', np.nanmin(dot_), np.nanmax(dot_), np.nanmean(dot_))#v_unit_vector_dsi_energy_channel_list[energy_i, channel_i])

In [ ]:
_data = (v_unit_vector_dsi_energy_channel_list[0, 0] * v_unit_vector_dsi_energy_channel_list[0, 0]).sum(dim='xyz')
for count in range(len(_data)):
    if _data[count].data < 0.99:
        print(_data[count].time)
        print('')
        print(_data[count].data)
        print('')

# 背景磁場ベクトルの決定

In [ ]:
import pyspedas as psp
import pytplot as pt
import xarray as xr
import numpy as np

psp.erg.mgf(trange=time_range_full, level='l2', datatype='256hz', coord='dsi', version='v03.03', no_update=True)
psp.erg.mgf(trange=time_range_full, level='l2', datatype='64hz', coord='dsi', version='v03.03', no_update=True)
psp.erg.mgf(trange=time_range_full, level='l2', datatype='8sec', coord='dsi', version='v03.03', no_update=True)

B_256Hz = pt.data_quants['erg_mgf_l2_mag_256hz_dsi'].rename({"v_dim": "xyz"})
B_64Hz = pt.data_quants['erg_mgf_l2_mag_64hz_dsi'].rename({"v_dim": "xyz"})
B_8sec  = pt.data_quants['erg_mgf_l2_mag_8sec_dsi'].rename({"v_dim": "xyz"})

In [ ]:
background_time_sec = 100 #[sec]

In [ ]:
B0_vector_dsi_energy_channel_list = {}
for energy_i in range(energy_num):
    for channel_i in range(channel_num):
        B_256Hz_interp       = B_256Hz.interp(time=v_unit_vector_dsi_energy_channel_list[energy_i, channel_i].time, method='linear')
        dt_B_256Hz_interp    = (B_256Hz_interp.time[1] - B_256Hz_interp.time[0]) / np.timedelta64(1, 's')
        B_background        = B_256Hz_interp.rolling(time=int(background_time_sec/dt_B_256Hz_interp), center=True).mean('time')

        B0_vector_dsi_energy_channel_list[energy_i, channel_i] = xr.DataArray(
            B_background.data,
            dims=['time', 'xyz'],
            coords={
                'time': B_background.time,
                'xyz': ['x', 'y', 'z'],
                'channel': channel_ax[channel_i],
                'energy_keV':   energy_ax[energy_i]
            },
            name=f'B0_vector_dsi_{energy_i}_{channel_i}'
        )
        print(f'B0_vector_dsi_energy_channel_list[{energy_i}, {channel_i}] = ', B0_vector_dsi_energy_channel_list[energy_i, channel_i])

# 対象とする垂直磁場成分を取得

In [ ]:
t_B_8sec    = B_8sec.time
dt_B_8sec = (t_B_8sec[2] - t_B_8sec[1]) / np.timedelta64(1, 's')
B_background    = B_8sec.rolling(time=int(background_time_sec/dt_B_8sec), center=True).mean('time')

In [ ]:
B_background_256Hz  = B_background.interp(time=B_256Hz.time, method='linear')
B_256Hz_perturb     = B_256Hz - B_background_256Hz
B_256Hz_perp = B_256Hz_perturb - (B_256Hz_perturb * B_background_256Hz).sum(dim='xyz') / (B_background_256Hz * B_background_256Hz).sum(dim='xyz') * B_background_256Hz

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, sosfiltfilt, sosfreqz

# ------------------ フィルタ設計 ------------------
fs = 256.                  # サンプリング周波数 [Hz]
order = 4                   # 4次 Butterworth（2-pole × 2-stage）

# 0.6 Hzから0.75 Hzの範囲のみを通すband-passフィルタ
lowcut = 0.6
highcut = 0.75

# btypeを'bandpass'に設定
sos = butter(N=order, Wn=[lowcut, highcut], btype='bandpass', fs=fs, output='sos')

# ------------------ インパルス応答 (変更なし) ------------------
n = 4096
delta = np.zeros(n)
delta[n//2] = 1

h = sosfiltfilt(sos, delta)
t = (np.arange(n) - n//2) / fs

# ------------------ 周波数応答 (変更なし) ------------------
w, H = sosfreqz(sos, worN=4096, fs=fs)
H_dbl = np.abs(H)**2

# ------------------ プロット (タイトルとハイライトを変更) ------------------
fig, axs = plt.subplots(2, 1, figsize=(10, 6), tight_layout=True)

# 時間領域
axs[0].plot(t, h)
axs[0].set_title('Impulse Response (Band-pass filter)')
axs[0].set_xlabel('Time [s]')
axs[0].set_ylabel('Amplitude')
axs[0].grid(True)

# 周波数領域
axs[1].semilogx(w, 20*np.log10(H_dbl), label='|H(f)|')

# 除去帯域を半透明のグレーで示す
axs[1].axvspan(lowcut, highcut, color='gray', alpha=0.3, label=f'{lowcut:.2f}-{highcut:.2f} Hz')
axs[1].axvline(0.6, color='red', lw=1)
axs[1].axvline(0.75, color='red', lw=1)

axs[1].set_title('Magnitude Response (Band-pass filter)')
axs[1].set_xlabel('Frequency [Hz]')
axs[1].set_ylabel('Magnitude [dB]')
axs[1].set_ylim(-20, 10)
axs[1].set_xlim(3E-1, 1)
axs[1].legend()
axs[1].grid(True, which='both', ls='--')

plt.show()

In [ ]:
import numpy as np
from scipy.signal import butter, sosfiltfilt
import pytplot as pt
import matplotlib.pyplot as plt
import os # osモジュールもインポートしておく

# フィルタパラメータ
fs = 256.                  # サンプリング周波数 [Hz]
lowcut = 0.60              # 通過域の下限周波数 [Hz]
highcut = 0.75            # 通過域の上限周波数 [Hz]
order = 4                 # フィルタの次数
window_sec = background_time_sec        # 背景磁場の移動平均窓幅 [sec]

sos = butter(N=order, Wn=[lowcut, highcut], btype='bandpass', fs=fs, output='sos')

def apply_filter_segmented(y, sos_mat):
    """NaN を含む 1‑D 配列にセグメントごとで sosfiltfilt を適用する"""
    good = np.isfinite(y)
    out  = np.full_like(y, np.nan)
    idx  = np.where(good)[0]
    segs = np.split(idx, np.where(np.diff(idx) != 1)[0] + 1)
    
    # パディング長はフィルタの次数に依存
    # scipyのドキュメントによると、sosfiltfiltのデフォルトpadlenは 3 * (sos.shape[1] // 2 - 1)
    # sosの形状は (n_sections, 6) なので、padlenは 3 * 2 = 6 となる
    padlen = 3 * (sos_mat.shape[1] - 1)
    
    for s in segs:
        if s.size > padlen:
            out[s] = sosfiltfilt(sos_mat, y[s])
    return out

B_256Hz_perp_bandpass = np.zeros(B_256Hz_perp.data.shape) * np.nan  # NaNで初期化
for i in range(3):
    B_256Hz_perp_bandpass[:, i] = apply_filter_segmented(B_256Hz_perp.data[:, i], sos)
da_B_256Hz_perp_bandpass = xr.DataArray(
    B_256Hz_perp_bandpass,
    dims=B_256Hz_perp.dims,
    coords=B_256Hz_perp.coords,
    name='B_256Hz_perp_bandpass'
)

da_B_256Hz_perp_bandpass_amp = np.sqrt((da_B_256Hz_perp_bandpass * da_B_256Hz_perp_bandpass).sum(dim='xyz'))

In [ ]:
import matplotlib as mpl

mpl.rcdefaults()
mpl.rcParams['font.size'] = 15

time_ax_range_min = np.datetime64('2017-11-15T16:17:00')
time_ax_range_max = np.datetime64('2017-11-15T16:22:00')

# plot
fig, ax = plt.subplots(1, 1, figsize=(10, 4), sharex=True)
ax.plot(da_B_256Hz_perp_bandpass_amp.time, da_B_256Hz_perp_bandpass_amp.data, lw=0.5, c='k')
ax.set_ylabel('[nT]')
ax.minorticks_on()
ax.grid(True, which='both', linestyle='--', alpha=0.5)
ax.set_xlim(time_ax_range_min, time_ax_range_max)
ax.set_ylim(0, 10)
plt.tight_layout()
plt.show()

# v_unitとB0、B_perpとのなす角を求めて、pitch angleとzeta angleをfluxデータに付与

In [ ]:
def ensure_xyz_coord(da):
    if 'xyz' in da.dims and 'xyz' not in da.coords:
        da = da.assign_coords(xyz=['x','y','z'])
    return da

flux_pitch_zeta_data_list    = {}
for energy_i in range(energy_num):
    for channel_i in range(channel_num):
        v_unit  = v_unit_vector_dsi_energy_channel_list[energy_i, channel_i]
        B0 = B0_vector_dsi_energy_channel_list[energy_i, channel_i].interp(time=v_unit.time)
        Bperp = da_B_256Hz_perp_bandpass.interp(time=v_unit.time)
        Bperp = Bperp - (Bperp * B0).sum(dim='xyz') / (B0 * B0).sum(dim='xyz') * B0

        v_unit  = ensure_xyz_coord(v_unit)
        B0      = ensure_xyz_coord(B0)
        Bperp   = ensure_xyz_coord(Bperp)

        dot_vB0 = (v_unit * B0).sum(dim='xyz')
        B0_2    = (B0 * B0).sum(dim='xyz')
        alpha = np.arccos(dot_vB0 / np.sqrt(B0_2))

        v_perp = v_unit - dot_vB0 / B0_2 * B0
        cross = xr.apply_ufunc(np.cross, Bperp, v_perp,
                               input_core_dims=[['xyz'], ['xyz']],
                               output_core_dims=[['xyz']], vectorize=True)
        Bperp_2     = (Bperp * Bperp).sum(dim='xyz')
        sin_zeta    = (cross * B0).sum(dim='xyz') / np.sqrt(B0_2 * Bperp_2)
        cos_zeta    = (v_perp * Bperp).sum(dim='xyz') / np.sqrt(Bperp_2)
        zeta = np.atan2(sin_zeta, cos_zeta)
        
        # 時間を統一（zeta基準）
        t = zeta['time']
        
        # flux の time を zeta に合わせる（必要なら補間）
        flux_ch = xr.DataArray(
            flux_data_arrays[energy_i][:, channel_i],
            coords={'time': flux_data_arrays[energy_i].coords['time']},  # ここは実データのtimeに合わせる
            dims=('time',)
        ).interp(time=t)
        
        da = xr.concat(
            [
                flux_ch.rename('differential_number_flux_keV'),
                np.rad2deg(alpha).rename('pitch_angle_deg'),
                (np.rad2deg(zeta) % 360.0).rename('zeta_angle_deg')
            ],
            dim='variable'
        ).assign_coords(variable=['differential_number_flux_keV','pitch_angle_deg','zeta_angle_deg']) \
         .transpose('time','variable')
        
        flux_pitch_zeta_data_list[energy_i, channel_i] = da.assign_coords(
            channel=channel_ax[channel_i],
            energy_keV=energy_ax[energy_i],
        )
        print(flux_pitch_zeta_data_list[energy_i, channel_i])

In [ ]:
time_ax_range_min = np.datetime64('2017-11-15T16:17:00')
time_ax_range_max = np.datetime64('2017-11-15T16:22:00')

for ch_idx in range(channel_num):
    data = flux_pitch_zeta_data_list[5, ch_idx].sel(time=slice(time_ax_range_min, time_ax_range_max))
    mask = (data[:, 0].values > 0) #& (data[:, 1].values > 168)
    data_mask = data[mask]
    if len(data_mask) > 0:
        print(energy_ax[5], channel_ax[ch_idx])
        print(np.nanmax(data_mask[:, 1]), np.nanmin(data_mask[:, 1]))
        print(np.nanmax(data_mask[:, 2]), np.nanmin(data_mask[:, 2]))
        print('')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.colors as mcolors
import matplotlib.cm as cm

time_ax_range_min = np.datetime64('2017-11-15T16:17:00')
time_ax_range_max = np.datetime64('2017-11-15T16:22:00')

for energy_i in range(energy_num):
    if energy_i != 5:
        continue

    # --- 1) 全channelでvmin/vmaxを決める（>0 かつ有限のみ） ---
    flux_vals = []
    for channel_i in range(channel_num):
        d = flux_pitch_zeta_data_list[energy_i, channel_i].sel(time=slice(time_ax_range_min, time_ax_range_max))
        f = d.sel(variable='differential_number_flux_keV').values
        flux_vals.append(f.ravel())
    flux_all = np.concatenate(flux_vals)
    mask = np.isfinite(flux_all) & (flux_all > 0)
    if not mask.any():
        continue
    vmin, vmax = flux_all[mask].min(), flux_all[mask].max()
    if np.log10(vmin) < np.log10(vmax) -2:
        vmin = vmax*1E-2
    norm = mcolors.LogNorm(vmin=vmin, vmax=vmax)
    cmap = 'turbo'

    # --- 2) 描画 ---
    fig = plt.figure(figsize=(10, 12))
    ax0 = fig.add_subplot(211)
    ax1 = fig.add_subplot(212)

    for channel_i in range(channel_num):
        #if channel_i != 0:
        #    continue
        d = flux_pitch_zeta_data_list[energy_i, channel_i].sel(time=slice(time_ax_range_min, time_ax_range_max))
        t = d['time'].values
        flux = d.sel(variable='differential_number_flux_keV').values
        alpha = d.sel(variable='pitch_angle_deg').values
        zeta  = d.sel(variable='zeta_angle_deg').values

        mask = ((alpha <= 11.25) | (alpha >= 167.75)) & (flux > 0)
        if np.any(mask):
            for ti, fi, ai, zi in zip(t[mask], flux[mask], alpha[mask], zeta[mask]):
                print(energy_i, channel_i, np.datetime_as_string(ti, unit='ns'), fi, ai, zi)

        ax0.scatter(t, alpha, c=flux, s=10, cmap=cmap, norm=norm, rasterized=True)
        ax1.scatter(t, zeta,  c=flux, s=10, cmap=cmap, norm=norm, rasterized=True)

    # 軸体裁
    ax0.set_ylabel(r'Pitch Angle $\alpha$' + '\n[deg]')
    ax1.set_ylabel(r'Phase difference $\zeta$' + '\n[deg]')

    ax0.set_title(f'LEP-i flux (energy = {energy_ax[energy_i]:.4f} keV)')
    for ax in (ax0, ax1):
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))
        ax.minorticks_on()
        ax.grid(True, which='both', linestyle='--', alpha=0.5)
        ax.set_xlim(time_ax_range_min, time_ax_range_max)
    ax0.set_ylim(0, 180);  ax0.set_yticks(np.arange(0, 181, 15))
    ax1.set_ylim(0, 360);  ax1.set_yticks(np.arange(0, 361, 30))

    # --- 3) カラーバーは共通norm/cmapから作る ---
    sm = cm.ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])  # 必須
    plt.colorbar(sm, ax=ax0, label='Differential number flux\n' + r'[$\mathrm{s}^{-1}\mathrm{cm}^{-2}\mathrm{str}^{-1}\mathrm{keV}^{-1}$]')
    plt.colorbar(sm, ax=ax1, label='Differential number flux\n' + r'[$\mathrm{s}^{-1}\mathrm{cm}^{-2}\mathrm{str}^{-1}\mathrm{keV}^{-1}$]')

    plt.tight_layout()
    plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.colors as mcolors
import matplotlib.cm as cm

time_ax_range_min = np.datetime64('2017-11-15T16:17:00')
time_ax_range_max = np.datetime64('2017-11-15T16:18:00')

for energy_i in range(energy_num):
    if energy_i != 5:
        continue

    # --- 1) 全channelでvmin/vmaxを決める（>0 かつ有限のみ） ---
    flux_vals = []
    for channel_i in range(channel_num):
        d = flux_pitch_zeta_data_list[energy_i, channel_i].sel(time=slice(time_ax_range_min, time_ax_range_max))
        f = d.sel(variable='differential_number_flux_keV').values
        flux_vals.append(f.ravel())
    flux_all = np.concatenate(flux_vals)
    mask = np.isfinite(flux_all) & (flux_all > 0)
    if not mask.any():
        continue
    vmin, vmax = flux_all[mask].min(), flux_all[mask].max()
    if np.log10(vmin) < np.log10(vmax) -2:
        vmin = vmax*1E-2
    norm = mcolors.LogNorm(vmin=vmin, vmax=vmax)
    cmap = 'turbo'

    # --- 2) 描画 ---


    for channel_i in range(channel_num):
        fig = plt.figure(figsize=(10, 12))
        ax0 = fig.add_subplot(211)
        ax1 = fig.add_subplot(212)
        d = flux_pitch_zeta_data_list[energy_i, channel_i].sel(time=slice(time_ax_range_min, time_ax_range_max))
        t = d['time'].values
        flux = d.sel(variable='differential_number_flux_keV').values
        alpha = d.sel(variable='pitch_angle_deg').values
        zeta  = d.sel(variable='zeta_angle_deg').values

        ax0.scatter(t, alpha, c=flux, s=10, cmap=cmap, norm=norm, rasterized=True)
        ax0.plot(t, alpha, lw=0.5, c='k')
        ax1.scatter(t, zeta,  c=flux, s=10, cmap=cmap, norm=norm, rasterized=True)
        ax1.plot(t, zeta, lw=0.5, c='k')

        # 軸体裁
        ax0.set_ylabel(r'Pitch Angle $\alpha$' + '\n[deg]')
        ax1.set_ylabel(r'Phase difference $\zeta$' + '\n[deg]')

        ax0.set_title(f'LEP-i flux (energy = {energy_ax[energy_i]:.4f} keV)')
        for ax in (ax0, ax1):
            ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))
            ax.minorticks_on()
            ax.grid(True, which='both', linestyle='--', alpha=0.5)
            ax.set_xlim(time_ax_range_min, time_ax_range_max)
        ax0.set_ylim(0, 180);  ax0.set_yticks(np.arange(0, 181, 10))
        ax1.set_ylim(0, 360);  ax1.set_yticks(np.arange(0, 361, 30))

        # --- 3) カラーバーは共通norm/cmapから作る ---
        sm = cm.ScalarMappable(norm=norm, cmap=cmap)
        sm.set_array([])  # 必須
        plt.colorbar(sm, ax=ax0, label='Differential number flux\n' + r'[$\mathrm{s}^{-1}\mathrm{cm}^{-2}\mathrm{str}^{-1}\mathrm{keV}^{-1}$]')
        plt.colorbar(sm, ax=ax1, label='Differential number flux\n' + r'[$\mathrm{s}^{-1}\mathrm{cm}^{-2}\mathrm{str}^{-1}\mathrm{keV}^{-1}$]')

        plt.tight_layout()
        plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.colors as mcolors
import matplotlib.cm as cm

time_ax_range_min = np.datetime64('2017-11-15T16:17:00')
time_ax_range_max = np.datetime64('2017-11-15T16:22:00')

for energy_i in range(energy_num):
    if energy_i != 5:
        continue

    # --- 1) 全channelでvmin/vmaxを決める（>0 かつ有限のみ） ---
    flux_vals = []
    alpha_vals = []
    for channel_i in range(channel_num):
        d = flux_pitch_zeta_data_list[energy_i, channel_i].sel(time=slice(time_ax_range_min, time_ax_range_max))
        f = d.sel(variable='differential_number_flux_keV').values
        alpha = d.sel(variable='pitch_angle_deg').values
        flux_vals.append(f.ravel())
        alpha_vals.append(alpha.ravel())
    flux_all = np.concatenate(flux_vals)
    alpha_all = np.concatenate(alpha_vals)
    mask = np.isfinite(flux_all) & (flux_all > 0) & (alpha_all > 125) & (alpha_all < 145)
    if not mask.any():
        continue
    vmin, vmax = flux_all[mask].min(), flux_all[mask].max()
    if np.log10(vmin) < np.log10(vmax) -2:
        vmin = vmax*1E-2
    norm = mcolors.LogNorm(vmin=vmin, vmax=vmax)
    cmap = 'turbo'

    # --- 2) 描画 ---
    fig = plt.figure(figsize=(10, 12))
    ax0 = fig.add_subplot(211)
    ax1 = fig.add_subplot(212)

    for channel_i in range(channel_num):
        #if channel_i != 3:
        #    continue
        d = flux_pitch_zeta_data_list[energy_i, channel_i].sel(time=slice(time_ax_range_min, time_ax_range_max))
        t = d['time'].values
        flux = d.sel(variable='differential_number_flux_keV').values
        alpha = d.sel(variable='pitch_angle_deg').values
        zeta  = d.sel(variable='zeta_angle_deg').values

        mask = (alpha > 125) & (alpha < 145) & (flux > 0)
        t_mask  = t[mask]
        flux_mask   = flux[mask]
        flux_mask_ratio = flux_mask# / vmax
        #norm = mcolors.Normalize(vmin=0.6, vmax=1.0)
        alpha_mask  = alpha[mask]
        zeta_mask   = zeta[mask]
        ax0.scatter(t_mask, alpha_mask, c=flux_mask_ratio, s=10, cmap=cmap, norm=norm, rasterized=True)
        ax1.scatter(t_mask, zeta_mask,  c=flux_mask_ratio, s=10, cmap=cmap, norm=norm, rasterized=True)

    # 軸体裁
    ax0.set_ylabel(r'Pitch Angle $\alpha$' + '\n[deg]')
    ax1.set_ylabel(r'Phase difference $\zeta$' + '\n[deg]')

    ax0.set_title(f'LEP-i flux (energy = {energy_ax[energy_i]:.4f} keV)')
    for ax in (ax0, ax1):
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))
        ax.minorticks_on()
        ax.grid(True, which='both', linestyle='--', alpha=0.5)
        ax.set_xlim(time_ax_range_min, time_ax_range_max)
    ax0.set_ylim(0, 180);  ax0.set_yticks(np.arange(0, 181, 15))
    ax1.set_ylim(0, 360);  ax1.set_yticks(np.arange(0, 361, 30))

    # --- 3) カラーバーは共通norm/cmapから作る ---
    sm = cm.ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])  # 必須
    plt.colorbar(sm, ax=ax0, label='Differential number flux / maximum')
    plt.colorbar(sm, ax=ax1, label='Differential number flux / maximum')

    plt.tight_layout()
    plt.show()


In [ ]:
import numpy as np
import xarray as xr
from scipy.stats import binned_statistic_2d
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.colors as mcolors
import matplotlib.cm as cm
from scipy.ndimage import convolve1d

time_ax_range_min = np.datetime64('2017-11-15T16:17:00')
time_ax_range_max = np.datetime64('2017-11-15T16:22:00')

pitch_angle_bins = np.linspace(0, 180, 17)   # 16 bins: 0,11.25,...,180
zeta_bins        = np.linspace(0, 360, 13)   # 12 bins: 0,30,...,360
dt = 8.0

time_bins = np.arange(
    time_ax_range_min.astype('datetime64[s]').astype('int64') - 2,
    time_ax_range_max.astype('datetime64[s]').astype('int64') + 6,
    dt
).astype('datetime64[s]')

def to_sec_i64(t):
    t = np.asarray(t).astype('datetime64[s]')
    return t.astype('int64')

def movmean_time_nan(A, win_sec, dt_sec=1):
    """
    NaN対応移動平均
    ----------
    A : 2D array (time × something)
    win_sec : 移動平均の半幅 [秒]
    dt_sec : Aの時間分解能 [秒]
    """
    # ±win_sec → 全体窓幅
    width = int(np.round(2 * win_sec / dt_sec)) + 1
    if width < 1:
        return A

    W = np.ones(width, dtype=float)
    valid = np.isfinite(A)
    A0 = np.where(valid, A, 0.0)

    # 値と有効数の1次元畳み込み（時間軸=0）
    num = convolve1d(A0, W, axis=0, mode='constant', cval=0.0)
    den = convolve1d(valid.astype(float), W, axis=0, mode='constant', cval=0.0)

    out = num / den
    out[den == 0] = np.nan
    return out

time_bins_sec = to_sec_i64(time_bins)

for energy_i in range(energy_num):
    if energy_i != 5:
        continue

    # ---- 全channelの点群を結合 ----
    all_t, all_alpha, all_zeta, all_flux = [], [], [], []
    for channel_i in range(channel_num):
        d     = flux_pitch_zeta_data_list[energy_i, channel_i].sel(time=slice(time_ax_range_min, time_ax_range_max))
        t     = d['time'].values
        flux  = d.sel(variable='differential_number_flux_keV').values
        alpha = d.sel(variable='pitch_angle_deg').values
        zeta  = d.sel(variable='zeta_angle_deg').values

        all_t.append(t)
        all_alpha.append(alpha)
        all_zeta.append(zeta)
        all_flux.append(flux)

    # 1D配列に
    t_cat    = np.concatenate(all_t)
    alpha_cat= np.concatenate(all_alpha).astype(float)
    zeta_cat = np.concatenate(all_zeta).astype(float)
    flux_cat = np.concatenate(all_flux).astype(float)

    # 有効データ
    good = np.isfinite(alpha_cat) & np.isfinite(zeta_cat) & np.isfinite(flux_cat) & (flux_cat > 0)
    if not np.any(good):
        continue

    tx_sec = to_sec_i64(t_cat[good])
    pa     = alpha_cat[good]
    ze     = zeta_cat[good]
    fv     = flux_cat[good]

    # ヤコビアン重み w = sin(alpha) # 立体角を考慮
    w = np.sin(np.deg2rad(pa))

    # ---- pitch(α)–time の重み付き平均 ----
    num_pa, xedges_pa, yedges_pa, _ = binned_statistic_2d(
        tx_sec, pa, fv * w, statistic='sum',
        bins=[time_bins_sec, pitch_angle_bins]
    )
    den_pa, _, _, _ = binned_statistic_2d(
        tx_sec, pa, w, statistic='sum',
        bins=[time_bins_sec, pitch_angle_bins]
    )
    hist_pitch = num_pa / den_pa
    hist_pitch[den_pa == 0] = np.nan

    # ---- zeta–time も同様（重みは同じ sin α）----
    num_ze, xedges_ze, yedges_ze, _ = binned_statistic_2d(
        tx_sec, ze, fv * w, statistic='sum',
        bins=[time_bins_sec, zeta_bins]
    )
    den_ze, _, _, _ = binned_statistic_2d(
        tx_sec, ze, w, statistic='sum',
        bins=[time_bins_sec, zeta_bins]
    )
    hist_zeta = num_ze / den_ze
    hist_zeta[den_ze == 0] = np.nan

    # ---- pcolormesh 用の中心値 ----
    tcent_sec = (xedges_pa[:-1] + xedges_pa[1:]) // 2
    time_cent = tcent_sec.astype('datetime64[s]')
    pa_cent   = 0.5 * (yedges_pa[:-1] + yedges_pa[1:])
    ze_cent   = 0.5 * (yedges_ze[:-1]  + yedges_ze[1:])

    # ---- 共通LogNorm（全点のfluxから） ----
    vmin, vmax = np.nanmin(fv), np.nanmax(fv)
    if vmin <= 0 or not np.isfinite(vmin) or not np.isfinite(vmax):
        continue
    # 2桁以上開いていれば下限を vmax*1e-2 に寄せる
    if np.log10(vmin) < np.log10(vmax) - 1.5:
        vmin = vmax * 10**-1.5
    norm_0 = mcolors.LogNorm(vmin=1E4, vmax=3E5)
    norm_1 = mcolors.LogNorm(vmin=7E4, vmax=3E5)
    #norm_0 = mcolors.LogNorm(vmin=vmin, vmax=vmax)
    #norm_1 = mcolors.LogNorm(vmin=vmin, vmax=vmax)
    cmap = 'turbo'

    # 出現数0ビンはNaNに
    cnt_pitch, _, _ = np.histogram2d(tx_sec, pa, bins=[time_bins_sec, pitch_angle_bins])
    cnt_zeta,  _, _ = np.histogram2d(tx_sec, ze, bins=[time_bins_sec, zeta_bins])
    hist_pitch[cnt_pitch == 0] = np.nan
    hist_zeta[cnt_zeta == 0]   = np.nan

    print(xedges_pa.astype('datetime64[s]'))

    # ---- 描画 ----
    fig = plt.figure(figsize=(10, 12))
    ax0 = fig.add_subplot(211)
    pcm0 = ax0.pcolormesh(
        xedges_pa.astype('datetime64[s]'), yedges_pa, hist_pitch.T, cmap=cmap, norm=norm_0, shading='auto'
    )
    ax0.set_ylabel(r'Pitch Angle $\alpha$' + '\n[deg]')
    ax0.set_title(f'LEP-i flux (energy = {energy_ax[energy_i]:.4f} keV)')
    ax0.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))
    ax0.minorticks_on()
    ax0.grid(True, which='both', linestyle='--', alpha=0.5)
    ax0.set_ylim(0, 180); ax0.set_yticks(np.arange(0, 181, 20))
    ax0.set_xlim(time_ax_range_min, time_ax_range_max)
    plt.colorbar(pcm0, ax=ax0, label='Differential number flux\n' + r'[$\mathrm{s}^{-1}\mathrm{cm}^{-2}\mathrm{str}^{-1}\mathrm{keV}^{-1}$]')

    ax1 = fig.add_subplot(212)
    pcm1 = ax1.pcolormesh(
        xedges_ze.astype('datetime64[s]'), yedges_ze, hist_zeta.T, cmap=cmap, norm=norm_1, shading='auto'
    )
    ax1.set_ylabel(r'Phase difference $\zeta$' + '\n[deg]')
    ax1.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))
    ax1.minorticks_on()
    ax1.grid(True, which='both', linestyle='--', alpha=0.5)
    ax1.set_ylim(0, 360); ax1.set_yticks(np.arange(0, 361, 30))
    ax1.set_xlim(time_ax_range_min, time_ax_range_max)
    plt.colorbar(pcm1, ax=ax1, label='Differential number flux\n' + r'[$\mathrm{s}^{-1}\mathrm{cm}^{-2}\mathrm{str}^{-1}\mathrm{keV}^{-1}$]')

    plt.tight_layout()
    plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.colors import Normalize

# ===== 積算関数 =====
def hist_zeta_timewin(t_s, ze, fw, ww,
                      time_bins, zeta_bins,
                      dt_bin=4.0, win_sec=60.0,
                      mode="backward"):
    """
    t_s: 時刻[s] float, ze: ζ[deg] in [0,360)
    fw: fv*sin(alpha)*dt_s, ww: sin(alpha)*dt_s
    return: hist_zeta (Nt, Nz), time_cent (Nt,)
    """
    time_bins_sec = time_bins.astype('datetime64[s]').astype('int64')
    Nt_out = len(time_bins_sec) - 1
    Nz = len(zeta_bins) - 1
    ze = np.mod(ze, 360.0)

    if mode == "backward":
        L = int(np.ceil(win_sec / dt_bin))
        t0_ext = time_bins_sec[0] - L*dt_bin
        t1_ext = time_bins_sec[-1]
    elif mode == "centered":
        Lh = int(np.ceil(0.5*win_sec / dt_bin))
        t0_ext = time_bins_sec[0] - Lh*dt_bin
        t1_ext = time_bins_sec[-1] + Lh*dt_bin
    else:
        raise ValueError("mode must be 'backward' or 'centered'.")

    edges_ext = np.arange(t0_ext, t1_ext + dt_bin, dt_bin).astype(float)
    Nt_ext = len(edges_ext) - 1

    k = np.floor((t_s - edges_ext[0]) / dt_bin).astype(int)
    valid = (k >= 0) & (k < Nt_ext)
    k = k[valid]; ze_v = ze[valid]
    fw_v = fw[valid].astype(float); ww_v = ww[valid].astype(float)

    j = np.digitize(ze_v, zeta_bins) - 1
    j = np.clip(j, 0, Nz-1)

    num = np.zeros((Nt_ext, Nz), float)
    den = np.zeros((Nt_ext, Nz), float)
    np.add.at(num, (k, j), fw_v)
    np.add.at(den, (k, j), ww_v)

    cs_num = np.vstack([np.zeros((1, Nz)), np.cumsum(num, axis=0)])
    cs_den = np.vstack([np.zeros((1, Nz)), np.cumsum(den, axis=0)])

    if mode == "backward":
        L = int(np.ceil(win_sec / dt_bin))
        idx0 = np.maximum(0, np.arange(Nt_ext) - L + 1)
        num_use = cs_num[1:] - cs_num[idx0]
        den_use = cs_den[1:] - cs_den[idx0]
        num_use = num_use[-Nt_out:]; den_use = den_use[-Nt_out:]
    else:
        Lh = int(np.ceil(0.5*win_sec / dt_bin))
        idx = np.arange(Nt_ext)
        i0 = np.clip(idx - Lh, 0, Nt_ext)
        i1 = np.clip(idx + Lh, 0, Nt_ext)
        num_c = cs_num[i1] - cs_num[i0]
        den_c = cs_den[i1] - cs_den[i0]
        start = Lh; stop = Lh + Nt_out
        num_use = num_c[start:stop]; den_use = den_c[start:stop]

    eps = 1e-12
    hist_zeta = num_use / den_use
    hist_zeta[den_use < eps] = np.nan

    time_cent = time_bins[:-1] + (time_bins[1:] - time_bins[:-1]) // 2
    return hist_zeta, time_cent

# ===== 入力 =====
time_ax_range_min = np.datetime64('2017-11-15T16:14:00')
time_ax_range_max = np.datetime64('2017-11-15T16:22:00')
time_ax_range_min_lim = np.datetime64('2017-11-15T16:17:00')
time_ax_range_max_lim = np.datetime64('2017-11-15T16:22:00')

zeta_bins = np.linspace(0, 360, 11)
dt_bin = 4.0
win_sec = 60.0
mode = "centered"

time_bins = np.arange(
    time_ax_range_min.astype('datetime64[s]').astype('int64') - 2,
    time_ax_range_max.astype('datetime64[s]').astype('int64') + 6,
    dt_bin
).astype('datetime64[s]')

def to_sec_float_ns(t_ns):
    return t_ns.astype('datetime64[ns]').astype('int64').astype(float) * 1e-9

for energy_i in range(energy_num):
    if energy_i != 5:
        continue

    # ---- データ結合 ----
    all_t, all_alpha, all_zeta, all_flux = [], [], [], []
    for ch in range(channel_num):
        d = flux_pitch_zeta_data_list[energy_i, ch].sel(time=slice(time_ax_range_min, time_ax_range_max))
        all_t.append(d['time'].values)
        all_flux.append(d.sel(variable='differential_number_flux_keV').values)
        all_alpha.append(d.sel(variable='pitch_angle_deg').values)
        all_zeta.append(d.sel(variable='zeta_angle_deg').values)

    t_cat     = np.concatenate(all_t)
    alpha_cat = np.concatenate(all_alpha).astype(float)
    zeta_cat  = np.concatenate(all_zeta).astype(float)
    flux_cat  = np.concatenate(all_flux).astype(float)

    good = (
        np.isfinite(alpha_cat) & np.isfinite(zeta_cat) & np.isfinite(flux_cat) &
        (flux_cat > 0) & (125 < alpha_cat) & (alpha_cat < 145)
    )
    if not np.any(good):
        continue

    # ---- 基本量 ----
    t_s = to_sec_float_ns(t_cat[good])
    pa  = alpha_cat[good]
    ze  = np.mod(zeta_cat[good], 360.0)
    fv  = flux_cat[good]

    order = np.argsort(t_s)
    t_s, pa, ze, fv = t_s[order], pa[order], ze[order], fv[order]

    dt_s = np.full_like(t_s, 0.015625, dtype=float)  # 15625 μs
    w  = np.sin(np.deg2rad(pa))
    fw = fv * w * dt_s
    ww = w  * dt_s

    # ===== ζ ヒートマップ（モード切替）=====
    hist_zeta, time_cent = hist_zeta_timewin(
        t_s, ze, fw, ww, time_bins, zeta_bins,
        dt_bin=dt_bin, win_sec=win_sec, mode=mode
    )

    # ===== プロット =====
    gs  = plt.figure(figsize=(10, 6)).add_gridspec(2, 20, hspace=0.05)
    ax0 = plt.gcf().add_subplot(gs[0, :19])
    ax1 = plt.gcf().add_subplot(gs[1, :19], sharex=ax0)
    cax = plt.gcf().add_subplot(gs[:, 19])

    ax0.plot(da_B_256Hz_perp_bandpass_amp.time,
             da_B_256Hz_perp_bandpass_amp.data, lw=0.5, c='k')
    ax0.set_ylabel(r'$|\mathbf{B}_{\mathrm{w}}|$' + '\n[nT]')
    ax0.set_title(f'energy = {energy_ax[energy_i]:.4f} keV, '
                  + r'$125\degree < \alpha < 145\degree$'
                  + f'  ({mode})')
    ax0.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))
    ax0.minorticks_on(); ax0.grid(True, which='both', linestyle='--', alpha=0.5)
    ax0.set_ylim(0, 10); ax0.set_yticks(np.arange(0, 10.1, 2))
    ax0.set_xlim(time_ax_range_min_lim, time_ax_range_max_lim)
    ax0.tick_params(labelbottom=False)

    # 正規化は表示範囲で固定
    mask = (time_cent >= time_ax_range_min_lim) & (time_cent <= time_ax_range_max_lim)
    vmax_ref = (np.nanmax(hist_zeta[mask, :])
                if np.any(mask) and np.any(np.isfinite(hist_zeta[mask, :]))
                else np.nanmax(hist_zeta))
    Z = (hist_zeta / vmax_ref).T
    pcm = ax1.pcolormesh(time_bins, zeta_bins, Z,
                         cmap='jet', norm=Normalize(vmin=0.6, vmax=1.0),
                         shading='auto')

    # ピーク ζ
    ze_cent = 0.5*(zeta_bins[:-1] + zeta_bins[1:])
    valid_col = np.any(np.isfinite(hist_zeta), axis=1)
    if np.any(valid_col):
        sub = hist_zeta[valid_col, :]
        ze_peak = ze_cent[np.nanargmax(sub, axis=1)]
        ax1.scatter(time_cent[valid_col], ze_peak, marker='x', s=30, c='k', lw=0.8, zorder=3)

    ax1.set_ylabel(r'$\zeta$' + '\n[deg]')
    ax1.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))
    ax1.minorticks_on(); ax1.grid(True, which='both', linestyle='--', alpha=0.5)
    ax1.set_ylim(0, 360); ax1.set_yticks(np.arange(0, 361, 45))
    ax1.set_xlim(time_ax_range_min_lim, time_ax_range_max_lim)

    plt.colorbar(pcm, cax=cax)
    plt.tight_layout()
    plt.show()
